# Read Iceberg via HMS

This scratchbook uses **PySpark + Iceberg** against your existing **Hive Metastore (HMS) pod**.
Aurora backs HMS metadata only — queries read **S3 Iceberg** files.

Replace `YOUR_DB` / `YOUR_TABLE` with a table your ingestion already registered.

In [ ]:
import os
from start_spark_session import get_spark, user_scratch_path

print("HMS_THRIFT_URI =", os.environ.get("HMS_THRIFT_URI"))
print("ICEBERG_WAREHOUSE =", os.environ.get("ICEBERG_WAREHOUSE"))
print("user scratch =", user_scratch_path())

spark = get_spark()
spark

In [ ]:
# List databases known to HMS
spark.sql("SHOW DATABASES").show(truncate=False)

In [ ]:
DB = os.environ.get("DEMO_DB", "YOUR_DB")
TABLE = os.environ.get("DEMO_TABLE", "YOUR_TABLE")

spark.sql(f"SHOW TABLES IN {DB}").show(truncate=False)

In [ ]:
df = spark.table(f"{DB}.{TABLE}")
df.printSchema()
df.show(20, truncate=False)
print("row estimate (limit count):", df.limit(1000).count())

In [ ]:
# Optional: simple pandas chart on a small sample (read-mostly)
sample = df.limit(500).toPandas()
sample.head()

## Write rules

- **Do not** `DROP` / overwrite production Iceberg tables from Light/Standard scratch profiles.
- Writes only to your scratch prefix: `user_scratch_path()`.

In [ ]:
# Example scratch write (Parquet), not a prod Iceberg commit
scratch = user_scratch_path().rstrip("/") + "/demo_sample.parquet"
df.limit(100).write.mode("overwrite").parquet(scratch)
print("wrote", scratch)